# Optimizer Comparison — Server Run

Upload this notebook to a vast.ai JupyterLab instance that has the optimizer-runner Docker image (or equivalent environment).  
**Working directory must be `/workspace`** (or wherever `src/` and `main.py` live).  

Sections:
1. Environment check
2. Configuration — **edit this cell**
3. Run regression
4. Run tabular classification
5. Run image classification
6. Results summary & inline plots

## 1. Environment check

In [ ]:
import os, sys, subprocess, importlib

# Ensure /workspace is on the path
WORKDIR = "/workspace"
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
print("cwd:", os.getcwd())
print("python:", sys.executable)

# GPU check
import torch
print(f"torch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"  GPU {i}: {props.name}  {mem_gb:.1f} GB")

# Key package versions
for pkg in ("pytorch_lightning", "ray", "mlflow", "optuna"):
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {m.__version__}")
    except ImportError:
        print(f"  {pkg}: NOT FOUND")

In [ ]:
# Install any missing packages (runs only if needed)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip error:", result.stderr[-2000:])
else:
    print("requirements OK")

# Clone optional optimizer repos if missing
for repo_url, dest in [
    ("https://github.com/nanowell/AdEMAMix-Optimizer-Pytorch", "AdEMAMix_Optimizer_Pytorch"),
    ("https://github.com/xinyuluo8561/Stacey", "Stacey"),
]:
    if not os.path.isdir(dest):
        print(f"Cloning {dest} ...")
        subprocess.run(["git", "clone", "--depth=1", repo_url, dest], check=True)
    if dest not in sys.path:
        sys.path.insert(0, os.path.join(WORKDIR, dest))
print("repos OK")

## 2. Configuration

Edit the values below before running.

In [ ]:
# ── EDIT THESE ───────────────────────────────────────────────────────────────

GPU_NUM       = torch.cuda.device_count() or 1   # GPUs to use (set 0 for CPU)
NUM_SAMPLES   = 40      # Optuna trials per optimizer
SEEDS         = [0, 1, 2, 3, 4]
NUM_WORKERS   = 4       # DataLoader workers
MOCK_RUN      = False   # True → quick smoke test (4 trials, 2 epochs, small data)

OUTPUT_DIR    = "/workspace/outputs"
MLFLOW_URI    = "/workspace/mlruns"
DATA_DIR      = "/workspace/data"
KAGGLE_JSON   = None    # e.g. "/root/.kaggle/kaggle.json" for Intel dataset

# Which tasks to run (comment out what you don't need)
RUN_REGRESSION            = True
RUN_TABULAR_CLASSIFICATION = True
RUN_IMAGE_CLASSIFICATION  = True

# Per-task overrides (leave 0 / [] to use defaults)
REGRESSION_DATASETS    = []   # [] → ["superconductivity", "yearmsd"]
REGRESSION_MODELS      = []   # [] → ["simple_mlp", "attention_mlp"]
TABULAR_DATASETS       = []   # [] → ["adult", "creditcard"]
TABULAR_MODELS         = []   # [] → ["simple_cls", "attention_cls"]
IMAGE_DATASETS         = []   # [] → ["places365", "intel"]
IMAGE_MODELS           = []   # [] → ["resnet18", "efficientnet_v2_s"]

# ─────────────────────────────────────────────────────────────────────────────
print(f"GPU_NUM={GPU_NUM}  NUM_SAMPLES={NUM_SAMPLES}  MOCK_RUN={MOCK_RUN}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
import logging
from pathlib import Path

# Logging: file + notebook output
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
log_path = f"{OUTPUT_DIR}/run.log"

fmt = "%(asctime)s  %(levelname)-8s  %(name)s  %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=fmt,
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(log_path, mode="a"),
    ],
    force=True,
)
for noisy in ("ray", "ray.tune", "ray.air", "pytorch_lightning", "lightning", "absl", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)

from src.runner import ExperimentConfig, run_experiments, TASK_DEFAULTS
print("imports OK")

## 3. Regression

In [ ]:
if RUN_REGRESSION:
    cfg_reg = ExperimentConfig(
        task_type   = "regression",
        datasets    = REGRESSION_DATASETS,
        model_types = REGRESSION_MODELS,
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = TASK_DEFAULTS["regression"]["batch_size"],
        num_epochs  = TASK_DEFAULTS["regression"]["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = OUTPUT_DIR,
        mlflow_uri  = MLFLOW_URI,
        data_dir    = DATA_DIR,
        num_workers = NUM_WORKERS,
    )
    print("Datasets:", cfg_reg.datasets)
    print("Models:  ", cfg_reg.model_types)
    reg_results = run_experiments(cfg_reg)
    print("Regression done.")
else:
    reg_results = {}
    print("Skipped.")

## 4. Tabular Classification

In [ ]:
if RUN_TABULAR_CLASSIFICATION:
    cfg_tab = ExperimentConfig(
        task_type   = "tabular_classification",
        datasets    = TABULAR_DATASETS,
        model_types = TABULAR_MODELS,
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = TASK_DEFAULTS["tabular_classification"]["batch_size"],
        num_epochs  = TASK_DEFAULTS["tabular_classification"]["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = OUTPUT_DIR,
        mlflow_uri  = MLFLOW_URI,
        data_dir    = DATA_DIR,
        num_workers = NUM_WORKERS,
    )
    print("Datasets:", cfg_tab.datasets)
    print("Models:  ", cfg_tab.model_types)
    tab_results = run_experiments(cfg_tab)
    print("Tabular classification done.")
else:
    tab_results = {}
    print("Skipped.")

## 5. Image Classification

In [ ]:
if RUN_IMAGE_CLASSIFICATION:
    cfg_img = ExperimentConfig(
        task_type   = "image_classification",
        datasets    = IMAGE_DATASETS,
        model_types = IMAGE_MODELS,
        num_samples = NUM_SAMPLES,
        seeds       = SEEDS,
        batch_size  = TASK_DEFAULTS["image_classification"]["batch_size"],
        num_epochs  = TASK_DEFAULTS["image_classification"]["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = MOCK_RUN,
        output_dir  = OUTPUT_DIR,
        mlflow_uri  = MLFLOW_URI,
        data_dir    = DATA_DIR,
        num_workers = NUM_WORKERS,
        kaggle_json = KAGGLE_JSON,
    )
    print("Datasets:", cfg_img.datasets)
    print("Models:  ", cfg_img.model_types)
    img_results = run_experiments(cfg_img)
    print("Image classification done.")
else:
    img_results = {}
    print("Skipped.")

## 6. Results summary

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path(OUTPUT_DIR) / "results"

# Cross-experiment summary (written by run_experiments)
cross_csv = results_dir / "cross_summary.csv"
if cross_csv.exists():
    df_cross = pd.read_csv(cross_csv, index_col=0)
    print("=== Cross-experiment summary ===")
    display(df_cross)
else:
    print("cross_summary.csv not found — run at least one task first.")

In [ ]:
# Per-experiment tuned summaries
for csv_path in sorted(results_dir.glob("*_tuned_summary.csv")):
    exp_key = csv_path.stem.replace("_tuned_summary", "")
    df = pd.read_csv(csv_path)
    print(f"\n=== {exp_key} — tuned summary ===")
    display(df)

In [ ]:
# Show saved plots inline
import glob
from IPython.display import Image, display as ipy_display

plot_subdirs = [
    "best_run_plots",
    "tuned_run_plots",
    "default_run_plots",
    "varying_rs_run_plots",
]

for subdir in plot_subdirs:
    pngs = sorted(glob.glob(f"{OUTPUT_DIR}/{subdir}/*.png"))
    if not pngs:
        continue
    print(f"\n{'='*60}")
    print(f"  {subdir}  ({len(pngs)} plots)")
    print(f"{'='*60}")
    for p in pngs:
        print(os.path.basename(p))
        ipy_display(Image(filename=p, width=900))

In [ ]:
# Telemetry analysis
from src.telemetry import load_telemetry, telemetry_summary

tel_dir = results_dir / "telemetry"
for jsonl_path in sorted(tel_dir.glob("*.jsonl")):
    exp_key = jsonl_path.stem.replace("_telemetry", "")
    df_tel = load_telemetry(str(jsonl_path))
    print(f"\n=== {exp_key} — telemetry ===")
    display(telemetry_summary(df_tel))

In [ ]:
# Start MLflow UI (accessible via vast.ai open port 5000)
import subprocess
proc = subprocess.Popen(
    ["mlflow", "ui", "--backend-store-uri", MLFLOW_URI, "--host", "0.0.0.0", "--port", "5000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("MLflow UI started on port 5000.")
print("Open  http://<instance-ip>:5000  in your browser.")
print("(Enable port 5000 in vast.ai instance settings if not already open.)")

## Utility: re-run a single experiment

Use this cell if you need to re-run just one `(dataset, model_type)` pair without re-running everything.

In [ ]:
# Example: re-run superconductivity + simple_mlp only
cfg_single = ExperimentConfig(
    task_type   = "regression",
    datasets    = ["superconductivity"],
    model_types = ["simple_mlp"],
    num_samples = NUM_SAMPLES,
    seeds       = SEEDS,
    batch_size  = TASK_DEFAULTS["regression"]["batch_size"],
    num_epochs  = TASK_DEFAULTS["regression"]["num_epochs"],
    gpu_num     = GPU_NUM,
    mock_run    = MOCK_RUN,
    output_dir  = OUTPUT_DIR,
    mlflow_uri  = MLFLOW_URI,
    data_dir    = DATA_DIR,
    num_workers = NUM_WORKERS,
)
# run_experiments(cfg_single)  # uncomment to run